<a href="https://colab.research.google.com/github/sathkumara-smna/258739K/blob/main/CapstoneLTE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Raw GitHub file URL
#url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/LTE_01_03_2026%20-%20Lite.xlsx"
url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/LTE%20Master%20Raw%20Data.xlsx"

# Load Excel file
df = pd.read_excel(url)

# Show first few rows
df.head()

In [ ]:
url1 = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/4G%20Site%20information.xlsx"

# Load into dataframe
info_4g = pd.read_excel(url1)
info_4g.head()


In [ ]:
url2 = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/LTE%20Traffic%20Bands.xlsx"

# Load into dataframe
ltetrafficbands = pd.read_excel(url2)
ltetrafficbands.head()

In [ ]:
df["BBU3900"] = 0
df["BBU3910"] = 0
df["rru_count"] = 0
df["lbbp_boards"] = 0
df["BBU3900BP"] = 0
df["BBU3910BP"] = 0
df["lbbp_bp"] = 0
df["ltetrfband"] = 0
df["ltetrfbandmax"] = 0
df["ltetrfbandextrapower"] = 0
df["calctotalpower"] = 0
df["rru_base_power"] = 180
df["rru_extra_band"] = 0
df["rru_extra_power"] = 0
df.head()

In [ ]:
# Merge BBU columns from info_4g using Site_ID
df = df.merge(
    info_4g[["Site_ID", "BBU 3900", "BBU 3910"]],
    on="Site_ID",
    how="left"
)

# Fill df columns
df["BBU3900"] = df["BBU 3900"].fillna(0).astype(int)
df["BBU3910"] = df["BBU 3910"].fillna(0).astype(int)

# Remove temporary columns
df.drop(columns=["BBU 3900", "BBU 3910"], inplace=True)

# Preview
df[["Site_ID", "BBU3900", "BBU3910"]].head()

In [ ]:
df = df.merge(
    info_4g[["Site_ID", "rru_count", "lbbp_boards"]],
    on="Site_ID",
    how="left",
    suffixes=("", "_from_4g")
)

# Fill values
df["rru_count"] = df["rru_count_from_4g"].fillna(0).astype(int)
df["lbbp_boards"] = df["lbbp_boards_from_4g"].fillna(0).astype(int)

# Clean up
df.drop(columns=["rru_count_from_4g", "lbbp_boards_from_4g"], inplace=True)

df.head()

In [ ]:
import numpy as np

df["BBU3900BP"] = np.where(
    df["rru_count"] == 0,
    0,
    (55 * df["BBU3900"] / df["rru_count"]).round(2)
)
df["BBU3910BP"] = np.where(
    df["rru_count"] == 0,
    0,
    (65 * df["BBU3910"] / df["rru_count"]).round(2)
)
df.head()

In [ ]:
import numpy as np

df["lbbp_bp"] = np.where(
    df["rru_count"] == 0,
    0,
    (42.5 * df["lbbp_boards"] / df["rru_count"]).round(2)
)
df[["lbbp_boards", "rru_count", "lbbp_bp"]].head()

In [ ]:
def get_lbbp_power(load):
    match = ltetrafficbands[
        (ltetrafficbands["Lower"] <= load) &
        (ltetrafficbands["Upper"] > load)
    ]

    if not match.empty:
        return match.iloc[0]["lbbp_power"]
    return 0  # default if no match

df["ltetrfband"] = df["traffic_load_mbps"].apply(get_lbbp_power)
df[["traffic_load_mbps", "ltetrfband"]].head()

In [ ]:
df.iloc[10:12]

In [ ]:
def get_upper_band(load):
    match = ltetrafficbands[
        (ltetrafficbands["Lower"] <= load) &
        (ltetrafficbands["Upper"] > load)
    ]

    if not match.empty:
        return match.iloc[0]["UpperValue"]
    return 0

df["ltetrfbandmax"] = df["traffic_load_mbps"].apply(get_upper_band)
df.iloc[10:12]

In [ ]:
# -------------------------------------------------
# Map rru_power from ltetrafficbands
# based on traffic_load_mbps range
# into df['rru_extra_band']
# -------------------------------------------------

def get_rru_power(traffic):

    match = ltetrafficbands[
        (traffic >= ltetrafficbands['Lower']) &
        (traffic <= ltetrafficbands['Upper'])
    ]

    if not match.empty:
        return match.iloc[0]['rru_power']

    # fallback for traffic above highest range
    return ltetrafficbands['rru_power'].max()

# Apply mapping
df['rru_extra_band'] = df['traffic_load_mbps'].apply(get_rru_power)

# Verify
df[['traffic_load_mbps', 'rru_extra_band']].head(5)

In [ ]:
df.iloc[10:12]

In [ ]:
# Calculate rru_extra_power

df['rru_extra_power'] = round(
    (
        df['traffic_load_mbps'] *
        df['rru_extra_band']
    ) / df['ltetrfbandmax'],
    2
)

# Verify
df[[
    'traffic_load_mbps',
    'rru_extra_band',
    'ltetrfbandmax',
    'rru_extra_power'
]].head(5)

In [ ]:
import numpy as np

df["ltetrfbandextrapower"] = np.where(
    (df["ltetrfbandmax"] == 0) | (df["rru_count"] == 0),
    0,
    (
        (df["traffic_load_mbps"] * df["ltetrfband"]) /
        (df["ltetrfbandmax"] * df["rru_count"])
    ).round(2)
)

In [ ]:
df[[
    "traffic_load_mbps",
    "ltetrfband",
    "ltetrfbandmax",
    "rru_count",
    "ltetrfbandextrapower"
]].head()

In [ ]:
df.iloc[1:2]

In [ ]:
df["calctotalpower"] = (
    df["BBU3900BP"] +
    df["BBU3910BP"] +
    df["lbbp_bp"] +
    df["rru_base_power"] +
    df["rru_extra_power"] +
    df["ltetrfbandextrapower"]
).round(2)

In [ ]:
df.head()

In [ ]:
#df.to_excel("full data.xlsx", index=False)

Maximum power of a Cell

In [ ]:
lte_max_cell = (
    df.groupby(["Site_ID", "Cell_ID"], as_index=False)["calctotalpower"]
    .max()
    .rename(columns={"calctotalpower": "max_cell_power"})
)

lte_max_cell.head()

In [ ]:
lte_max_cell.to_excel("lte_max_cell.xlsx", index=False)

Add Maximum power of all cells per site

In [ ]:
lte_max_power_site = (
    lte_max_cell.groupby("Site_ID", as_index=False)["max_cell_power"]
    .sum()
    .rename(columns={"max_cell_power": "max_power"})
)

# Round to 2 decimal points
lte_max_power_site["max_power"] = (
    lte_max_power_site["max_power"].round(2)
)

lte_max_power_site.head()

In [ ]:
lte_max_power_site.to_excel("lte_max_power_site.xlsx", index=False)